In [7]:
'''
parsing incoming json data

When a client like a frontend app or postman sends data to the FastAPI server
it usually sends in json format, fastapi automatically converts json into a python object
using pydantic model, and it ensures the incoming data has the right field and correct data types

'''

from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

# Define structure of incoming data
class Item(BaseModel):
    name: str
    price: float
    description: str = None
    tax: float = None

# Endpoint to receive JSON
@app.post("/items/")
async def create_item(item: Item):
    return {
        "name": item.name,
        "price": item.price
    }

In [8]:
{
  "name": "Laptop",
  "price": 1000.0
}

'''

If wrong input (like price = "cheap")

{
  "detail": [
    {
      "loc": ["body", "price"],
      "msg": "value is not a valid float"
    }
  ]
}

'''

{'name': 'Laptop', 'price': 1000.0}

In [10]:
'''
Customizing HTTP REQUESTS

By default, fastapi sends a simple response with status code 200 ok, but in real world
apis, often want more control - like sending 201 when something is created
adding extra metadata or even custom  headers, fastapi allows you to fully customize what goes
back to the client, the status code, the response body and even the headers

'''
from fastapi import FastAPI,status

app = FastAPI()

@app.post("/items/",status_code=status.HTTP_201_CREATED)
async def create_item(item:Item):
  return {"name":item.name,"price":item.price}



In [12]:
'''
customing response body like adding timestamp,metadata,etc
'''
from fastapi.responses import JSONResponse
from datetime import datetime

@app.post("/items/")
async def create_item(item:Item):
  response_data = {
        "timestamp": datetime.now().isoformat(),
        "data": {
            "name": item.name,
            "price": item.price
        }
  }
  return JSONResponse(content=response_data)


In [14]:
'''

Using HTTPException for Error Hanlding

In real applications, things go wrong, data may not exist input may be invalid etc,
Instead of crashing or sending unclear responses
FASTAPI uses HTTPException to return clean and meaninful error with proper status code
we raise these exceptions when something is wrong, think of it like stopping execution and sending a clear error message to client

'''

from fastapi import FastAPI,HTTPException
app = FastAPI()
items = {
    "1":{"name":"Item1","price":10}
}
@app.get("/items/{item_id}")
async def read_item(item_id: str):
    if item_id not in items:
        raise HTTPException(
            status_code=404,
            detail="Item not found"
        )
    return items[item_id]

